![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 3 -- Lab 4: Fine-tune a Pretrained Model

The most common move in transfer learning: take a model trained on millions of
images, **replace its final classifier** with a new one for *your* classes, and
train only that part. It's fast and works even with little data.

*Tip: use a GPU runtime (Runtime -> Change runtime type -> GPU) for speed.*

## 1) Setup

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset

## 2) Load a pretrained model and freeze it

Freezing means we keep everything it already learned and won't change it.

In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
for p in model.parameters():
    p.requires_grad = False              # freeze the backbone

## 3) Replace the classifier

ResNet was built for 1000 classes. We swap its last layer for one with **10** outputs (CIFAR-10). Only this new layer will train.

In [ ]:
model.fc = nn.Linear(model.fc.in_features, 10)   # new, trainable classifier
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

## 4) Get the data (CIFAR-10)

We use small subsets to keep training quick.

In [ ]:
prep = weights.transforms()
train = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=prep)
test  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=prep)
train_dl = DataLoader(Subset(train, range(2000)), batch_size=32, shuffle=True)
test_dl  = DataLoader(Subset(test,  range(500)),  batch_size=32)

## 5) Train only the new classifier

In [ ]:
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)   # only the new layer
loss_fn = nn.CrossEntropyLoss()
model.train()
for epoch in range(2):
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss_fn(model(x), y).backward()
        opt.step()
    print(f"epoch {epoch + 1} done")

## 6) Check accuracy on unseen images

In [ ]:
model.eval(); correct = total = 0
with torch.no_grad():
    for x, y in test_dl:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
print(f"accuracy: {correct / total:.1%}")

## Try it yourself

You trained only the final layer in minutes and still got useful accuracy --
because the frozen backbone already knows how to *see*. Try training for more
epochs, or unfreeze a few more layers.

### *Contributed By: Sattam Altwaim*